# Lab Monte Carlo — agent-assisted, human-verified

Helper functions for sampling, rank correlation, S-curves, and tornado charts. Use an AI assistant to scaffold your model, then verify it here.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ri128-rutgers/quality-management-fall2026/blob/main/lab_monte_carlo.ipynb) · Also opens in Azure ML Studio notebooks or any Jupyter; nothing here needs a GPU.

**How to use:** run every cell top to bottom once. Cells marked `# >>> YOUR WORK` are where you change inputs or add interpretation. Keep your AI-assistant transcript if you used one; the verification log asks what you checked.

In [ ]:
# Setup — works in Colab and locally
import os, sys, io, urllib.request
REPO = 'ri128-rutgers/quality-management-fall2026'
DATA_BASE = f'https://raw.githubusercontent.com/{REPO}/main/' if 'google.colab' in sys.modules or not os.path.exists('../data') else '../data/'
if 'google.colab' in sys.modules:
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/{REPO}/main/tvm.py', 'tvm.py')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tvm import *
pd.options.display.float_format = '{:,.2f}'.format
def load(name): return pd.read_csv(DATA_BASE + name)
print('data from', DATA_BASE)

In [ ]:
from scipy import stats
rng = np.random.default_rng(7)

def tri(low, mode, high, n): return rng.triangular(low, mode, high, n)
def pert(low, mode, high, n, lam=4):
    a = 1 + lam*(mode-low)/(high-low); b = 1 + lam*(high-mode)/(high-low)
    return low + rng.beta(a, b, n)*(high-low)
def lognorm(median, sigma, n): return rng.lognormal(np.log(median), sigma, n)

def impose_rank_corr(samples: dict, pairs: list[tuple[str,str,float]]):
    """Iman–Conover style: reorder columns to hit target Spearman correlations (approximate, pairwise)."""
    out = {k: np.sort(v) for k, v in samples.items()}
    n = len(next(iter(out.values())))
    base = rng.standard_normal(n)
    ranks = {k: stats.rankdata(rng.standard_normal(n)) for k in out}
    for a, b, rho in pairs:
        za = stats.norm.ppf(ranks[a]/(n+1)); zb = rho*za + np.sqrt(1-rho**2)*rng.standard_normal(n)
        ranks[b] = stats.rankdata(zb)
    return {k: out[k][ranks[k].astype(int)-1] for k in out}

def s_curve(total, point_estimate=None, title='Discounted total cost'):
    xs = np.sort(total); ps = np.arange(1, len(xs)+1)/len(xs)
    plt.plot(xs, ps); plt.xlabel('$'); plt.ylabel('cumulative probability'); plt.title(title)
    if point_estimate is not None:
        plt.axvline(point_estimate, color='k', ls='--'); print('P(cost <= point estimate) =', round((xs <= point_estimate).mean(), 3))
    for q in (0.5, 0.7, 0.8): print(f'P{int(q*100)} = {np.quantile(xs, q):,.0f}')
    plt.show()

def tornado(samples: dict, total):
    rho = {k: stats.spearmanr(v, total).correlation for k, v in samples.items()}
    s = pd.Series(rho).sort_values(key=abs); s.plot.barh(); plt.title('Spearman correlation with total'); plt.show(); return s

## Your model

Ask your assistant for a Monte Carlo version of your Case 2 cost element structure; paste it below and make it use the helpers. The example is deliberately thin — an assistant will produce something like it, and it has at least three problems you should catch (hint: which inputs are independent that should not be; which distribution has an impossible tail; is the point estimate the mean or the mode?).

In [ ]:
n = 10_000
s = {
    'dev_pm': lognorm(110, 0.35, n),
    'rate': tri(14_500, 16_200, 18_500, n),
    'sensors': pert(280, 320, 400, n),
    'cloud_per_sensor_month': tri(22, 31, 45, n),
    'sustain_share': tri(0.08, 0.16, 0.30, n),
    'retrain_events_yr': rng.integers(1, 4, n),
    'retrain_cost': tri(12_000, 18_000, 30_000, n),
    'downtime_reduction': tri(0.15, 0.27, 0.40, n),
}
s = impose_rank_corr(s, [('dev_pm', 'rate', 0.4), ('sensors', 'cloud_per_sensor_month', 0.3)])   # >>> YOUR WORK: justify these pairs and rhos
r = 0.08; N = 10
dev = s['dev_pm']*s['rate']; hw = s['sensors']*395 + 14*2_750
annual = s['sensors']*s['cloud_per_sensor_month']*12 + s['sustain_share']*dev + s['retrain_events_yr']*s['retrain_cost']
total = dev + hw + annual*p_a(r, N) + hw*0.35*(p_f(r,4)+p_f(r,8))
point = 110*16_200 + 320*395 + 14*2_750 + (320*31*12 + 0.16*110*16_200 + 2*18_000)*p_a(r,N)
s_curve(total, point)
rank = tornado(s, total)
print('mean', round(total.mean()), ' sd', round(total.std()), ' CV', round(total.std()/total.mean(),3))

In [ ]:
# Convergence check
for k in (500, 2_000, 5_000, 10_000):
    print(k, 'mean', round(total[:k].mean()), ' P80', round(np.quantile(total[:k], 0.8)))
# Bayesian update (normal-normal): prior from the simulation, likelihood from a year-1 actual
prior_mu, prior_sd = total.mean(), total.std()
actual_y1_total_scaled = point * 1.12    # >>> YOUR WORK: replace with the year-1 actual from the lab data note in Canvas, scaled to total
obs_sd = prior_sd * 0.6
post_var = 1/(1/prior_sd**2 + 1/obs_sd**2); post_mu = post_var*(prior_mu/prior_sd**2 + actual_y1_total_scaled/obs_sd**2)
print('posterior mean', round(post_mu), ' posterior P80', round(stats.norm.ppf(0.8, post_mu, np.sqrt(post_var))))